In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score,roc_curve,RocCurveDisplay
from sklearn.ensemble import GradientBoostingClassifier
from tqdm import tqdm
import os
os.chdir('E:/Machine_Learning/Cases/human-resources-analytics/')

In [6]:
df = pd.read_csv('HR_comma_sep.csv')
df.head()


,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,Department,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.10,0.77,6,247,4,0,1,0,sales,low
3,0.92,0.85,5,259,5,0,1,0,sales,low
4,0.89,1.00,5,224,5,0,1,0,sales,low


In [8]:
X , y = df.drop('left',axis =1), df['left']
X_train,X_test,y_train,y_test= train_test_split(X,y,test_size=0.3,random_state=26,stratify=y)

ohe = OneHotEncoder(sparse_output= False,drop='first')
ct = make_column_transformer((ohe,make_column_selector(dtype_include='object')),
                            ('passthrough',make_column_selector(dtype_include=['int64','float64'])),
                            verbose_feature_names_out=False)
ct = ct.set_output(transform='pandas')
X_train_ohe = ct.fit_transform(X_train)
X_test_ohe = ct.transform(X_test)

In [10]:
learn_rates = np.linspace(0.01,0.9,10)
max_depths = [None,2,3,4]
n_estimators = [25,50,100]
scores = []
for l in tqdm(learn_rates):
    for m in max_depths:
        for n in n_estimators:
            gbm = GradientBoostingClassifier(random_state=26,learning_rate=l,max_depth=m,n_estimators=n)
            gbm.fit(X_train_ohe,y_train)
            y_pred_prob = gbm.predict_proba(X_test_ohe)
            scores.append([l,m,n,roc_auc_score(y_test,y_pred_prob[:,1])])
df_scores = pd.DataFrame(scores,columns=['Learning_rate','Max_depth','n_estimators','roc_auc_score'])
df_scores.sort_values(by='roc_auc_score',ascending=False)

100%|██████████| 10/10 [01:56<00:00, 11.60s/it]


,Learning_rate,Max_depth,n_estimators,roc_auc_score
34,0.207778,4.0,50,0.994069
35,0.207778,4.0,100,0.993173
85,0.702222,NaN,50,0.992989
86,0.702222,NaN,100,0.992989
23,0.108889,4.0,100,0.992928
...,...,...,...,...
4,0.010000,2.0,50,0.957106
5,0.010000,2.0,100,0.956285
3,0.010000,2.0,25,0.952598
118,0.900000,4.0,50,0.895149


## XGBoost

In [11]:
from xgboost import XGBClassifier

In [13]:
learn_rates = np.linspace(0.01,0.9,10)
max_depths = [None,2,3,4]
n_estimators = [25,50,100]
scores = []
for l in tqdm(learn_rates):
    for m in max_depths:
        for n in n_estimators:
            xgb = XGBClassifier(random_state=26,learning_rate=l,max_depth=m,n_estimators=n)
            xgb.fit(X_train_ohe,y_train)
            y_pred_prob = xgb.predict_proba(X_test_ohe)
            scores.append([l,m,n,roc_auc_score(y_test,y_pred_prob[:,1])])
df_scores = pd.DataFrame(scores,columns=['Learning_rate','Max_depth','n_estimators','roc_auc_score'])
df_scores.sort_values(by='roc_auc_score',ascending=False)

100%|██████████| 10/10 [00:19<00:00,  1.91s/it]


,Learning_rate,Max_depth,n_estimators,roc_auc_score
85,0.702222,NaN,50,0.994366
86,0.702222,NaN,100,0.994201
14,0.108889,NaN,100,0.993776
69,0.504444,4.0,25,0.993705
46,0.306667,4.0,50,0.993663
...,...,...,...,...
6,0.010000,3.0,25,0.975420
15,0.108889,2.0,25,0.973712
5,0.010000,2.0,100,0.957833
4,0.010000,2.0,50,0.956294


## LightBoosting

In [14]:
from lightgbm import LGBMClassifier

In [16]:
learn_rates = np.linspace(0.01,0.9,10)
max_depths = [None,2,3,4]
n_estimators = [25,50,100]
scores = []
for l in tqdm(learn_rates):
    for m in max_depths:
        for n in n_estimators:
            lgb = XGBClassifier(random_state=26,learning_rate=l,max_depth=m,n_estimators=n)
            lgb.fit(X_train_ohe,y_train)
            y_pred_prob = lgb.predict_proba(X_test_ohe)
            scores.append([l,m,n,roc_auc_score(y_test,y_pred_prob[:,1])])
df_scores = pd.DataFrame(scores,columns=['Learning_rate','Max_depth','n_estimators','roc_auc_score'])
df_scores.sort_values(by='roc_auc_score',ascending=False)

100%|██████████| 10/10 [00:19<00:00,  1.94s/it]


,Learning_rate,Max_depth,n_estimators,roc_auc_score
85,0.702222,NaN,50,0.994366
86,0.702222,NaN,100,0.994201
14,0.108889,NaN,100,0.993776
69,0.504444,4.0,25,0.993705
46,0.306667,4.0,50,0.993663
...,...,...,...,...
6,0.010000,3.0,25,0.975420
15,0.108889,2.0,25,0.973712
5,0.010000,2.0,100,0.957833
4,0.010000,2.0,50,0.956294


## CatBoosting

In [17]:
from catboost import CatBoostClassifier

In [19]:
learn_rates = np.linspace(0.01,0.9,10)
max_depths = [None,2,3,4]
n_estimators = [25,50,100]
scores = []
for l in tqdm(learn_rates):
    for m in max_depths:
        for n in n_estimators:
            cat = CatBoostClassifier(random_state=26,learning_rate=l,verbose=False,max_depth=m,n_estimators=n)
            cat.fit(X_train_ohe,y_train)
            y_pred_prob = cat.predict_proba(X_test_ohe)
            scores.append([l,m,n,roc_auc_score(y_test,y_pred_prob[:,1])])
df_scores = pd.DataFrame(scores,columns=['Learning_rate','Max_depth','n_estimators','roc_auc_score'])
df_scores.sort_values(by='roc_auc_score',ascending=False)

100%|██████████| 10/10 [00:35<00:00,  3.56s/it]


,Learning_rate,Max_depth,n_estimators,roc_auc_score
38,0.306667,NaN,100,0.993234
86,0.702222,NaN,100,0.993132
37,0.306667,NaN,50,0.992795
26,0.207778,NaN,100,0.992682
35,0.207778,4.0,100,0.992341
...,...,...,...,...
7,0.010000,3.0,50,0.966821
6,0.010000,3.0,25,0.946160
5,0.010000,2.0,100,0.944676
4,0.010000,2.0,50,0.937260


## with using cat_features of CatBoosting

In [20]:
df.dtypes

satisfaction_level       float64
last_evaluation          float64
number_project             int64
average_montly_hours       int64
time_spend_company         int64
Work_accident              int64
left                       int64
promotion_last_5years      int64
Department                   str
salary                       str
dtype: object

In [25]:
df = pd.read_csv('HR_comma_sep.csv')
X , y = df.drop('left',axis =1), df['left']
X_train,X_test,y_train,y_test= train_test_split(X,y,test_size=0.3,random_state=26,stratify=y)

cate = ['Department','salary']
learn_rates = np.linspace(0.01,0.9,10)
max_depths = [None,2,3,4]
n_estimators = [25,50,100]
scores = []
for l in tqdm(learn_rates):
    for m in max_depths:
        for n in n_estimators:
            cat = CatBoostClassifier(random_state=26,learning_rate=l,verbose=False,max_depth=m,n_estimators=n,cat_features=cate)
            cat.fit(X_train,y_train)
            y_pred_prob = cat.predict_proba(X_test)
            scores.append([l,m,n,roc_auc_score(y_test,y_pred_prob[:,1])])
df_scores = pd.DataFrame(scores,columns=['Learning_rate','Max_depth','n_estimators','roc_auc_score'])
df_scores.sort_values(by='roc_auc_score',ascending=False)

100%|██████████| 10/10 [02:32<00:00, 15.22s/it]


,Learning_rate,Max_depth,n_estimators,roc_auc_score
62,0.504444,NaN,100,0.992526
38,0.306667,NaN,100,0.992415
71,0.504444,4.0,100,0.992128
37,0.306667,NaN,50,0.991954
61,0.504444,NaN,50,0.991906
...,...,...,...,...
7,0.010000,3.0,50,0.968339
6,0.010000,3.0,25,0.955532
5,0.010000,2.0,100,0.941407
4,0.010000,2.0,50,0.938260
